In [2]:
!pip install torch 

  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/204.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/204.2 MB ? eta -:--:--
   ---------------------------------------- 0.8/204.2 MB 3.0 MB/s eta 0:01:07
   ---------------------------------------- 1.3/204.2 MB 2.8 MB/s eta 0:01:13
   ---------------------------------------- 1.6/204.2 MB 2.6 MB/s eta 0:01:18
   ---------------------------------------- 2.1/204.2 MB 2.3 MB/s eta 0:01:27
    --------------------------------------- 2.9/204.2 MB 2.5 MB/s eta 0:01:22
    --------------------------------------- 3.7/204.2 MB 2.7 MB/s eta 0:01:15
    --------------------------------------- 4.5/204.2 MB 2.8 MB/s eta 0:01:12
   - -------------------------------------- 5.2/204.2 MB 2.9 MB/s eta 0:01:09
   - -------------------------------------- 


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\praga\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import torch
from transformers import BertTokenizer, BertModel
import pandas as pd

# Load Pretrained BERT Model (PyTorch version)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.eval()  # Set to evaluation mode

# Student Details Table (Simulated Dataset)
data = {
    'Student_ID': [101, 102, 103],
    'Name': ['Alice', 'Bob', 'Charlie'],
    'Age': [20, 21, 22],
    'Course': ['Computer Science', 'Mathematics', 'Physics'],
    'Marks': [85, 78, 92],
    'Grade': ['A', 'B', 'A+']
}
df = pd.DataFrame(data)

# Convert table to searchable text format (context for BERT)
student_texts = [
    f"{row['Name']} is {row['Age']} years old, enrolled in {row['Course']}, scored {row['Marks']} marks, and has grade {row['Grade']}."
    for _, row in df.iterrows()
]
all_student_context = " ".join(student_texts)  # Combine all student info into one context string

# Process Query with BERT
def get_bert_response(query):
    # Combine query with student context
    input_text = f"Question: {query} Context: {all_student_context}"
    inputs = tokenizer(input_text, return_tensors='pt', padding=True, truncation=True, max_length=512)
    
    with torch.no_grad():
        outputs = bert_model(**inputs)
    # Use the [CLS] token embedding (not used for logic, just to process input)
    outputs.last_hidden_state[:, 0, :].detach().cpu()  # Shape: [1, 768]
    
    # Keyword-based response generation
    query_lower = query.lower()
    
    # Check for student name in query
    name = None
    for student_name in df['Name']:
        if student_name.lower() in query_lower:
            name = student_name
            break
    
    if 'marks' in query_lower and name:
        marks = df[df['Name'] == name]['Marks'].values[0]
        return f"{name} scored {marks} marks."
    if 'course' in query_lower and name:
        course = df[df['Name'] == name]['Course'].values[0]
        return f"{name} is enrolled in {course}."
    if 'highest' in query_lower and 'physics' in query_lower:
        physics_students = df[df['Course'] == 'Physics']
        max_marks = physics_students['Marks'].max()
        top_student = physics_students[physics_students['Marks'] == max_marks]['Name'].values[0]
        return f"{top_student} scored the highest in Physics with {max_marks} marks."
    if 'grade a' in query_lower:
        a_students = df[df['Grade'] == 'A'][['Name', 'Marks', 'Course']].to_string(index=False)
        return f"Students with Grade A:\n{a_students}"
    
    # Default response: full info if name is found, otherwise all context
    if name:
        return student_texts[df[df['Name'] == name].index[0]]
    return "I couldn’t find specific info. Here’s all student data: " + all_student_context

# Chatbot Interface (CLI)
def chatbot():
    print("Welcome to the Student Query Chatbot! (Type 'exit' to quit)")
    while True:
        query = input("Enter your query: ").strip()
        if query.lower() == 'exit':
            print("Goodbye!")
            break
        if not query:
            print("Please enter a valid query.")
            continue
        
        response = get_bert_response(query)
        print(f"Answer: {response}")

# Run the chatbot
if __name__ == "__main__":
    chatbot()

ModuleNotFoundError: No module named 'torch'